# NFL passing-yard predictions and Kalshi backtest with Nori

Run every cell from a fresh Python 3.11 Jupyter or Colab environment. This
notebook downloads public nflverse football data, reconstructs the blog's
feature transformations, runs fresh Nori predictions for all 18 regular-season
weeks of 2025, and optionally downloads Kalshi quotes to replay its strategy.
**It never downloads precomputed research predictions or requires an internal repository.**
The first run computes predictions; subsequent runs can reuse verified outputs
generated locally from matching inputs, model, configuration, and runtime.

The original feature rules, pruning, model weights, and betting decisions are
ported here—not the earlier simplified baseline. Results below are measured
from the downloaded inputs. Public nflverse files can be revised: a fresh
2020 source differs from the original cached snapshot, so matching the original
24.1% number is a result to verify, not a value to hard-code.

CPU works but the full run can take hours. A GPU is optional. To smoke-test,
change WEEKS to (1,); that is a partial run, not the full-season reproduction.
Kalshi is enabled by default; an API outage does not discard your forecasts.

## 1. Install the measured package versions
    Run this before importing the packages. If Jupyter already imported different
    versions, restart its kernel after installation and run again.

In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

required = {
    "synthefy-nori": "0.19.0", "nflreadpy": "0.1.5", "numpy": "2.4.6",
    "pandas": "3.0.5", "polars": "1.44.0", "torch": "2.13.0",
    "matplotlib": "3.11.1", "requests": "2.34.2", "huggingface-hub": "1.28.0",
    "scipy": "1.17.1", "scikit-learn": "1.9.0", "pyarrow": "25.0.1",
}
missing = []
for package, wanted in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != wanted:
        missing.append(f"{package}=={wanted}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print({name: metadata.version(name) for name in required})

## 2. Download the public implementation
    The feature transformations are too long for one readable notebook cell.
    Their adjacent source files are fetched from a fixed public Git commit and
    checked against SHA-256 hashes. Downloading this notebook alone is sufficient.
    The downloads are ordinary source code, not data or precomputed forecasts.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import requests

SOURCE_REF = 'c5e6bd4bd9176886efb00b29edc3c84f387c07fd'
SOURCE_HASHES = {'nfl_blog_inference.py': '9cfbda4feb5296901b91d0015161a38011480500b5dcbe65ae42067504f648c9', 'nfl_blog_strategy.py': 'e9e7ed81de73cd0a051c47abe9c39cd7b844ec89bab3f6782a183c7600d9af99', 'nfl_passing_yards_markets.py': 'fa01b67bddfad81404a6e467710a2549c8baae0bcf9706b31f0894976dbbaa52', 'nfl_blog_features/__init__.py': '4ae0cb6dd2b6f60e805f493f75962fa05ebbc88312bb7f70522282861d08bdfa', 'nfl_blog_features/availability_features.py': '19937dd2cbf88238cb8bea4182fb1890a73b1b30952d08ba87f098c1f935c594', 'nfl_blog_features/checkpoint_history.py': '8f164cb8ef97d7e02792d7a9a26207470faab1a6525c6a942760da8a3d418a46', 'nfl_blog_features/context_features.py': 'e2995081484ff48b3c7aba28612e51f92d0e93aca7f31bbfd8ec70153e0c0efd', 'nfl_blog_features/data.py': 'acb7736e0ad1be1e5a10c4bf30091e0f2f16fe4c0bd40eb802b6d30d94b5611d', 'nfl_blog_features/defense_features.py': 'cb59a8d2f6ced271707874dd3f5ff7237bd5f9a1f7250c7ed98cab2100400478', 'nfl_blog_features/derived_features.py': 'df61a2c9b3cb0368fc5be5a1fd38190a2d743684cd65bcc917c05576eacccc6c', 'nfl_blog_features/feature_sets.py': 'e820c55b4ab664a6d50bd252ee4a47083151205fab911b16fb6fa291475c1335', 'nfl_blog_features/features.py': 'b221114cdf3ea11eb41806ba15e31097c1b9075444a6936e8128ca28196d9e7a', 'nfl_blog_features/live_features.py': 'c11cd761aed941bb65b391985a33281376da25067ebcf5015f35d7b90d5fdfcf', 'nfl_blog_features/live_flow_features.py': '10d73f7c27c1a21e5702d2c7f53eb4b90d2f64b7ff21c4873b23efa22e782d32', 'nfl_blog_features/live_q1_features.py': '48b355b6e7e0ee2198f54880f3445d4bb66072cbd0455918cd1e808b9d1e2085', 'nfl_blog_features/offense_features.py': '2e58c9c1fad720909a8ad2cb62b98906150b39141f39f62867ba2dfe4aad1803', 'nfl_blog_features/pipeline.py': '7dc1d9d50259980d1717c22d992c243c92821d27ff76f041690cb07ef8514760', 'nfl_blog_features/season_context_features.py': '133cb4a40a8589ff952ab81b8ef7d78cd19daea1d7613b0a3a41d77282197778'}
SOURCE_DIR = Path("nfl_blog_source") / SOURCE_REF
base_url = f"https://raw.githubusercontent.com/Synthefy/synthefy-nori/{SOURCE_REF}/examples/notebooks"
for filename, expected_hash in SOURCE_HASHES.items():
    destination = SOURCE_DIR / filename
    if not destination.exists():
        response = requests.get(f"{base_url}/{filename}", timeout=60)
        response.raise_for_status()
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(response.content)
    if hashlib.sha256(destination.read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Helper hash mismatch: {destination}; remove this file and retry")
sys.path.insert(0, str(SOURCE_DIR.resolve()))
print(f"Verified {len(SOURCE_HASHES)} public source files at {SOURCE_REF}")

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display
from nfl_blog_features import build_blog_features, blog_feature_columns
from nfl_blog_inference import run_blog_predictions, prediction_metrics, modeling_rows
from nfl_blog_strategy import run_kalshi_backtest, probability_over_line, summarize

CACHE_DIR = Path(os.environ.get("NFL_CACHE_DIR", "nfl_blog_cache"))
OUTPUT_DIR = Path(os.environ.get("NFL_OUTPUT_DIR", "nfl_blog_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEEKS = tuple(range(1, 19))  # All regular-season weeks; use (1,) only for a smoke test.
DEVICE = os.environ.get("NFL_DEVICE", "cpu")  # For example: "cuda:0".
RUN_MARKET_BACKTEST = True
EXECUTION_COST = 0.05  # Additional modeled cost per one-contract purchase.
REFRESH_SOURCE_DATA = False
RESUME_PREDICTIONS = True  # Reuse only matching, fingerprinted local computations.
print({"weeks": WEEKS, "device": DEVICE, "kalshi": RUN_MARKET_BACKTEST})

## 3. Pull football data and build the two checkpoint tables
    nflreadpy downloads schedules, player statistics, depth charts, participation,
    play-by-play, and team metadata. Full-game rolling form uses 2016–2017 as
    warmup. Model context starts in 2018 and expands only with earlier weeks.

    The 140 numeric candidates include QB form, offense, defense, game/season
    context, game state at the checkpoint, and the QB's matching Q1/halftime
    history. The original selected model uses identity to compute individual
    history; it does **not** feed categorical QB IDs or weather to Nori.

    Q1 uses the earliest timestamped Q2 record as its boundary, excluding that
    record. Halftime uses the last Q2 record. The decision is two minutes later.
    Finalized play data may include corrections and do not prove feed arrival time.

In [ ]:
features = build_blog_features(CACHE_DIR, OUTPUT_DIR, refresh=REFRESH_SOURCE_DATA)
overview = pd.DataFrame([
    {"checkpoint": h, "rows": table.height,
     "eligible_2025_rows": modeling_rows(table).filter(pl.col("season") == 2025).height,
     "candidate_features": len(blog_feature_columns())}
    for h, table in features.items()
])
display(overview)

In [ ]:
example_columns = ["game_id", "week", "actual_qb_name", "live_decision_utc",
                   "live_qb_passing_yards", "live_qb_attempts", "official_passing_yards"]
history_columns = [c for c in blog_feature_columns() if c.startswith("checkpoint_history_")][:3]
display(features["q1"].filter(pl.col("season") == 2025)
        .select(example_columns + history_columns).head(8).to_pandas())
print("All candidate feature names:")
print("\n".join(blog_feature_columns()))

## 4. Run Nori—fresh predictions, week by week
    For each week and checkpoint, rank candidate features using correlations
    computed on past context only. Drop a feature when its absolute correlation
    with an already-kept feature exceeds 0.75. Use the entire retained context:
    no subsampling or different lightweight model is substituted.

    The pinned public Nori checkpoint predicts **remaining** yards. Adding the
    yards already thrown converts every quantile into a final-yard prediction.
    fit() stores the context; it does not train new model weights. Each week's
    predictions and chosen columns are saved as they finish.
    On a rerun, RESUME_PREDICTIONS checks local provenance fingerprints before
    reusing a completed week. Changed inputs, selected features, model, runtime,
    or device trigger fresh inference. Set it to False to recompute every week.

In [ ]:
predictions = run_blog_predictions(
    features["q1"], features["halftime"], OUTPUT_DIR,
    weeks=WEEKS, device=DEVICE, season=2025, resume=RESUME_PREDICTIONS,
)

## 5. Measure forecast error and coverage
    MAE and RMSE below use the predictive median. Pinball loss measures quantile
    accuracy; P10–P90 coverage is the fraction of final totals inside the predicted
    middle 80%. These are forecast metrics, not betting returns.

In [ ]:
metrics = {h: prediction_metrics(frame) for h, frame in predictions.items()}
display(pd.DataFrame(metrics).T.drop(columns=["pinball_loss"]))
display(pd.DataFrame({h: values["pinball_loss"] for h, values in metrics.items()}).rename_axis("quantile"))
(OUTPUT_DIR / "forecast_metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
display(predictions["q1"].select(
    "week", "actual_qb_name", "live_qb_passing_yards", "nori_p10",
    "nori_median", "nori_p90", "official_passing_yards", "context_rows", "selected_feature_count"
).head(12).to_pandas())

## 6. See the distribution in 5-yard bins
    Prefer the blog's Burrow example when that week is included; otherwise use the
    first predicted row. This plot is computed from this run's quantiles. Values
    are not copied from the blog. Bar heights approximate probability mass by
    interpolating the predictive CDF at half-yard bin boundaries.

In [ ]:
example = predictions["q1"].filter(
    (pl.col("game_id") == "2025_13_CIN_BAL") & (pl.col("actual_qb_name") == "Joe Burrow")
)
row = (example if example.height else predictions["q1"].head(1)).to_dicts()[0]
line = 200
taus = row["nori_quantile_taus"]
values = row["nori_quantile_values"]
max_yards = max(500, int(np.ceil(max(values) / 5) * 5))
boundaries = np.arange(-0.5, max_yards + 5, 5)
cdf = 1 - np.array([probability_over_line(taus, values, x) for x in boundaries])
mass = np.maximum(0, np.diff(cdf))
centers = boundaries[:-1] + 2.5
p_over = probability_over_line(taus, values, line - 0.5)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(centers, 100 * mass, width=4.5,
       color=np.where(centers >= line, "#33856b", "#c98a66"))
ax.axvline(line - 0.5, color="#b42318", label=f"{line}+ yards; Nori probability {p_over:.1%}")
ax.set(xlabel="Final passing yards", ylabel="Probability per 5-yard bin (%)",
       title=f"{row['actual_qb_name']} — Week {row['week']} — after Q1")
ax.legend()
plt.show()
fig.savefig(OUTPUT_DIR / "passing_yards_distribution.png", bbox_inches="tight")

## 7. Replay the original Kalshi rule
    After Q1, select the quoted line whose midpoint is closest to 50¢, then its
    higher-edge side. Buy only if probability × $1 − price − fee is at least 10¢.
    If Q1 does not buy, select the maximum-edge halftime candidate. Its edge must
    reach 10¢ and its probability for the **same line and side** must be at least
    the Q1 probability. At most one contract is bought per quarterback-game.

    Use the latest minute-close bid/ask between the anchor and decision; exclude
    quotes over five minutes old or with spreads over 10¢. YES buys at the YES ask;
    NO buys at one minus the YES bid. Modeled taker fees and the extra execution
    allowance reduce reported profit. Settlement comes from the public market.
    These are **quote-based simulations, not verified fills**.

    Missing API responses stop only the market stage. They never become invented
    quotes, inferred fills, or a profitable result on an incomplete download.

In [ ]:
if RUN_MARKET_BACKTEST:
    decisions, market_report = run_kalshi_backtest(
        predictions["q1"], predictions["halftime"], CACHE_DIR,
        execution_cost=EXECUTION_COST,
    )
else:
    decisions = pl.DataFrame()
    market_report = {"status": "disabled", "roi": None}
print(json.dumps(market_report, indent=2))
(OUTPUT_DIR / "kalshi_status.json").write_text(json.dumps(market_report, indent=2))

In [ ]:
if market_report["status"] == "quote_based_simulation":
    decisions.write_parquet(OUTPUT_DIR / "kalshi_decisions.parquet")
    trades = decisions.filter(pl.col("bet_taken"))
    display(trades.select(
        "week", "actual_qb_name", "selected_horizon", "line", "side",
        "yes_bid", "yes_ask", "model_probability", "entry_price", "fee",
        "expected_net_edge", "stressed_capital", "stressed_profit"
    ).to_pandas())
    trades.write_csv(OUTPUT_DIR / "kalshi_selected_trades.csv")
    sensitivities = pd.DataFrame([summarize(decisions, cost) for cost in (0.0, 0.05, 0.10)])
    display(sensitivities[["execution_cost", "bets", "capital", "pnl", "roi"]])
else:
    print("Forecasts are saved. No betting return is reported without available market data.")

## 8. Keep the provenance with the results
    Feature input hashes, pinned model identity, per-week feature selections,
    forecast metrics, and the market status are saved beside predictions. The
    original strategy was refined on 2025: this is an exploratory reproduction,
    not an untouched test or a guarantee of future returns. Public source revisions
    and runtime differences can change numbers even with the same algorithm.

In [ ]:
run_manifest = {
    "source_ref": SOURCE_REF, "source_hashes": SOURCE_HASHES,
    "package_versions": {p: metadata.version(p) for p in required},
    "weeks": WEEKS, "device": DEVICE, "execution_cost": EXECUTION_COST,
    "market_status": market_report["status"],
    "resume_verified_local_predictions": RESUME_PREDICTIONS,
    "precomputed_research_predictions": False,
    "input_hash_file": "feature_input_hashes.json",
}
(OUTPUT_DIR / "notebook_run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
display(pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size}
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))
print(f"Outputs: {OUTPUT_DIR.resolve()}")